# Memory AI Lab — Boundary Detector Training

**Workflow :** éditer le code dans VS Code → `git push` → ouvrir ce notebook dans [colab.research.google.com](https://colab.research.google.com)

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Ce que fait ce notebook

```
Stage 1 — Boundary Detector
   Input  : embeddings mE5-base (768d) + gap temporel
   Modèle : BoundaryMLP 770 → 128 → 32 → 1
   Output : P(frontière) pour chaque message
   Durée  : ~2 min CPU | ~20 sec GPU
```

**Données requises sur Google Drive (`memory_ai_data/`) :**
```
group_anon.txt
group_gold_tune.json
group_gold_test.json
```

Le modèle entraîné est sauvegardé dans `memory_ai_data/boundary_detector.pt`.

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!python -m spacy download fr_core_news_sm -q
print('✓ OK')
print()
print('⚠️  Si première exécution : Exécution > Redémarrer la session,')
print('   puis relancer à partir de la cellule 3.')

In [ ]:
# ── CELLULE 3 : Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/memory_ai_data'
print(f'✓ Drive monté → {DATA_DIR}')

In [ ]:
# ── CELLULE 4 : Parse + Embeddings mE5-base (GPU + cache) ─────────────────
import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from parsers.whatsapp_parser import parse_whatsapp_chat

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

# mE5-base — multilingual, 768d, handles FR/EN code-switching
MODEL_NAME  = 'intfloat/multilingual-e5-base'
PREFIX      = 'passage: '   # requis par mE5 pour les documents
EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings_me5.npy'

all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [PREFIX + a.content for a in all_artifacts]
print(f'[1/2] {len(texts)} messages parsés')

if EMBED_CACHE.exists():
    all_embeddings = np.load(EMBED_CACHE)
    assert len(all_embeddings) == len(texts), 'Cache périmé — supprimer group_embeddings_me5.npy'
    print('[2/2] Embeddings chargés depuis cache')
else:
    print(f'[2/2] Calcul sur {device} (mE5-base 768d) ...')
    model = SentenceTransformer(MODEL_NAME, device=device)
    all_embeddings = model.encode(
        texts, batch_size=256, show_progress_bar=True,
        device=device, convert_to_numpy=True
    ).astype(np.float32)
    np.save(EMBED_CACHE, all_embeddings)
    print(f'      Sauvegardé → {EMBED_CACHE}')

print(f'      Shape : {all_embeddings.shape}  (attendu : (n, 768))')

In [ ]:
# ── CELLULE 5 : Charger tune / test ───────────────────────────────────────
import json

def load_split(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    n = len(data['artifacts'])
    y_true = [None] * n
    for ep in data['episodes']:
        for idx in range(ep['start_idx'], ep['end_idx'] + 1):
            if idx < n:
                y_true[idx] = ep['episode_id']
    return data['artifacts'], y_true, data['episodes'], data['meta']

tune_arts, y_true_tune, tune_eps, tune_meta = load_split(f'{DATA_DIR}/group_gold_tune.json')
test_arts, y_true_test, test_eps, test_meta = load_split(f'{DATA_DIR}/group_gold_test.json')

n_tune = len(tune_arts)
arts_tune = all_artifacts[:n_tune]
arts_test = all_artifacts[n_tune:n_tune + len(test_arts)]
emb_tune  = all_embeddings[:n_tune]
emb_test  = all_embeddings[n_tune:n_tune + len(test_arts)]

print(f'✓ Tune : {len(tune_eps)} épisodes · {n_tune} msgs · {tune_meta["period"]}')
print(f'✓ Test : {len(test_eps)} épisodes · {len(test_arts)} msgs · {test_meta["period"]}')

In [ ]:
# ── CELLULE 6 : Extraction features + labels ──────────────────────────────
from boundary_detector import extract_features, extract_labels

X_tune = extract_features(emb_tune, arts_tune)
y_tune = extract_labels(y_true_tune, len(arts_tune))

X_test = extract_features(emb_test, arts_test)
y_test = extract_labels(y_true_test, len(arts_test))

n_pos_tune = int(y_tune.sum())
n_neg_tune = len(y_tune) - n_pos_tune

print(f'Tune features : {X_tune.shape}  (attendu : (n, 770))')
print(f'Labels tune   : {n_pos_tune} frontières · {n_neg_tune} continuations')
print(f'Ratio         : 1:{n_neg_tune // max(n_pos_tune, 1)} → pos_weight={n_neg_tune // max(n_pos_tune, 1)}')

In [ ]:
# ── CELLULE 7 : Entraînement BoundaryMLP ──────────────────────────────────
from boundary_detector import BoundaryDetector

detector = BoundaryDetector(device=device)
detector.fit(X_tune, y_tune, n_epochs=30, lr=1e-3, batch_size=512, verbose=True)

print('\n✓ Entraînement terminé')

In [ ]:
# ── CELLULE 8 : Optimisation seuil sur TUNE ───────────────────────────────
# IMPORTANT : optimiser sur tune uniquement, jamais sur test
best_thr = detector.optimize_threshold(X_tune, y_tune)

probs_tune = detector.predict_proba(X_tune)
preds_tune = (probs_tune >= best_thr).astype(int)

tp = int(((preds_tune == 1) & (y_tune == 1)).sum())
fp = int(((preds_tune == 1) & (y_tune == 0)).sum())
fn = int(((preds_tune == 0) & (y_tune == 1)).sum())
prec = tp / (tp + fp + 1e-8)
rec  = tp / (tp + fn + 1e-8)
f1   = 2 * prec * rec / (prec + rec + 1e-8)

print(f'\nMétriques sur TUNE :')
print(f'  Précision  : {prec:.4f}')
print(f'  Rappel     : {rec:.4f}')
print(f'  F1         : {f1:.4f}')
print(f'  Frontières détectées : {preds_tune.sum()} / {int(y_tune.sum())} gold')

In [ ]:
# ── CELLULE 9 : Évaluation sur TEST (une seule fois) ──────────────────────
# Ne pas re-tuner après avoir vu ce score

import numpy as np

probs_test = detector.predict_proba(X_test)
preds_test = (probs_test >= detector.threshold).astype(int)

tp = int(((preds_test == 1) & (y_test == 1)).sum())
fp = int(((preds_test == 1) & (y_test == 0)).sum())
fn = int(((preds_test == 0) & (y_test == 1)).sum())
prec = tp / (tp + fp + 1e-8)
rec  = tp / (tp + fn + 1e-8)
f1   = 2 * prec * rec / (prec + rec + 1e-8)

n_boundaries_test = preds_test.sum()
n_total_test = len(preds_test)

print(f"""
╔══════════════════════════════════════════════════╗
║  BOUNDARY DETECTOR — Résultat TEST               ║
╠══════════════════════════════════════════════════╣
║  Précision  : {prec:.4f}                         ║
║  Rappel     : {rec:.4f}                         ║
║  F1         : {f1:.4f}                          ║
╠══════════════════════════════════════════════════╣
║  Frontières : {n_boundaries_test} / {int(y_test.sum())} gold (sur {n_total_test} msgs)    ║
║  Taux       : {n_boundaries_test/n_total_test:.1%} des messages sont frontières    ║
║  Réduction  : {n_total_test/max(n_boundaries_test,1):.1f}x moins d'appels AttachScore       ║
╚══════════════════════════════════════════════════╝
""")

In [ ]:
# ── CELLULE 10 : Sauvegarde ────────────────────────────────────────────────
DETECTOR_PATH = f'{DATA_DIR}/boundary_detector.pt'
detector.save(DETECTOR_PATH)
print(f'✓ Modèle sauvegardé → {DETECTOR_PATH}')
print(f'  Seuil : {detector.threshold:.2f}')
print()
print('Étape suivante : ouvrir 01_eval_ari.ipynb et activer use_hybrid=True')